In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 92.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## Local Inference on GPU
Model page: https://huggingface.co/ibm-granite/granite-3.3-2b-instruct

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/ibm-granite/granite-3.3-2b-instruct)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [2]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="ibm-granite/granite-3.3-2b-instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

...

added_tokens.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': 'I am an assistant designed to help answer your questions and provide information. I strive to provide concise and accurate responses in English.'}]}]

In [3]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("ibm-granite/granite-3.3-2b-instruct")
model = AutoModelForCausalLM.from_pretrained("ibm-granite/granite-3.3-2b-instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

I am an assistant designed to help answer your questions and provide information. I strive to provide concise and accurate responses in English.<|end_of_text|>


In [4]:
!pip install -q gradio transformers torch black autopep8 reportlab requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.9/88.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 29.0 MB/s eta 0:00:00


In [7]:
import gradio as gr
from datetime import datetime
from pathlib import Path
import ast
import re
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak, Preformatted
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
import urllib.parse

# ============================================================================
# DOC-GENIE v3 — DEEP CONTEXTUAL ANALYSIS ENGINE
# ============================================================================

class FunctionAnalysis:
    """Rich structured result from deep AST analysis."""
    def __init__(self):
        self.name = ""
        self.params = []
        self.return_type = "Any"

        # Control flow
        self.has_for_loop = False
        self.has_while_loop = False
        self.has_conditions = False
        self.has_early_return = False
        self.has_recursion = False
        self.is_recursive = False

        # Error handling
        self.raises_exceptions = []      # list of exception names raised
        self.has_try_except = False
        self.caught_exceptions = []

        # Data patterns
        self.has_list_comp = False
        self.has_dict_comp = False
        self.has_set_comp = False
        self.has_generator = False
        self.has_yield = False           # generator function

        # Operations detected
        self.arithmetic_ops = set()     # add, sub, mul, div, mod, pow, floor_div
        self.comparison_ops = set()     # eq, lt, gt, etc.
        self.boolean_ops = set()        # and, or, not
        self.bitwise_ops = set()

        # Structural patterns
        self.calls_builtins = set()     # sum, len, sorted, map, filter, etc.
        self.calls_external = set()     # non-builtin function calls
        self.accesses_attributes = []   # obj.attr patterns → hints about input types
        self.string_ops = False         # .format, f-strings, .split, .join etc.
        self.dict_access = False        # d[key] or d.get(...)
        self.list_mutations = False     # .append, .extend, .pop, .insert

        # Return analysis
        self.return_expressions = []    # what actually gets returned
        self.always_returns = False
        self.returns_boolean = False
        self.returns_collection = False
        self.returns_self = False       # for builder pattern

        # Decorator hints
        self.decorators = []

        # Inferred purpose
        self.purpose_tags = set()       # e.g. 'validator', 'transformer', 'calculator', 'sorter'


PYTHON_BUILTINS = {
    'abs', 'all', 'any', 'bin', 'bool', 'callable', 'chr', 'dict', 'dir',
    'divmod', 'enumerate', 'eval', 'filter', 'float', 'format', 'frozenset',
    'getattr', 'hasattr', 'hash', 'hex', 'id', 'input', 'int', 'isinstance',
    'issubclass', 'iter', 'len', 'list', 'map', 'max', 'min', 'next', 'object',
    'oct', 'open', 'ord', 'pow', 'print', 'range', 'repr', 'reversed', 'round',
    'set', 'setattr', 'slice', 'sorted', 'str', 'sum', 'super', 'tuple', 'type',
    'vars', 'zip',
}

OP_NAMES = {
    ast.Add: 'add', ast.Sub: 'sub', ast.Mult: 'mul', ast.Div: 'div',
    ast.Mod: 'mod', ast.Pow: 'pow', ast.FloorDiv: 'floor_div',
    ast.MatMult: 'matmul',
    ast.Eq: 'eq', ast.NotEq: 'neq', ast.Lt: 'lt', ast.LtE: 'lte',
    ast.Gt: 'gt', ast.GtE: 'gte', ast.Is: 'is', ast.IsNot: 'is_not',
    ast.In: 'in', ast.NotIn: 'not_in',
    ast.BitAnd: 'bitand', ast.BitOr: 'bitor', ast.BitXor: 'bitxor',
    ast.LShift: 'lshift', ast.RShift: 'rshift',
}


class DocGenieAnalyzer:
    """Deep AST-based analyzer producing rich, contextually accurate docstrings."""

    # ── Signature Extraction ───────────────────────────────────────────────

    @staticmethod
    def extract_signature(code: str):
        """Parse the function definition and return (signature_dict, func_def_node)."""
        try:
            tree = ast.parse(code)
            func_def = next(
                (n for n in ast.walk(tree) if isinstance(n, ast.FunctionDef)),
                None
            )
            if func_def is None:
                return None, None

            params = []
            defaults_offset = len(func_def.args.args) - len(func_def.args.defaults)

            for i, arg in enumerate(func_def.args.args):
                param_type = "Any"
                if arg.annotation:
                    try:
                        param_type = ast.unparse(arg.annotation)
                    except Exception:
                        param_type = "Any"

                default = None
                default_idx = i - defaults_offset
                if default_idx >= 0:
                    try:
                        default = ast.unparse(func_def.args.defaults[default_idx])
                    except Exception:
                        default = "..."

                params.append({'name': arg.arg, 'type': param_type, 'default': default})

            # *args and **kwargs
            if func_def.args.vararg:
                a = func_def.args.vararg
                t = ast.unparse(a.annotation) if a.annotation else "Any"
                params.append({'name': f"*{a.arg}", 'type': t, 'default': None, 'variadic': True})

            if func_def.args.kwarg:
                a = func_def.args.kwarg
                t = ast.unparse(a.annotation) if a.annotation else "Any"
                params.append({'name': f"**{a.arg}", 'type': t, 'default': None, 'variadic': True})

            return_type = "None"
            if func_def.returns:
                try:
                    return_type = ast.unparse(func_def.returns)
                except Exception:
                    return_type = "Any"
            else:
                return_type = None   # unknown, will be inferred

            decorators = []
            for d in func_def.decorator_list:
                try:
                    decorators.append(ast.unparse(d))
                except Exception:
                    pass

            sig = {
                'name': func_def.name,
                'params': params,
                'return_type': return_type,
                'decorators': decorators,
            }
            return sig, func_def

        except SyntaxError:
            return None, None
        except Exception:
            return None, None

    # ── Deep Logic Analysis ────────────────────────────────────────────────

    @staticmethod
    def analyze(func_def, signature: dict) -> FunctionAnalysis:
        fa = FunctionAnalysis()
        fa.name = signature['name']
        fa.params = signature['params']
        fa.return_type = signature.get('return_type')
        fa.decorators = signature.get('decorators', [])

        param_names = {p['name'].lstrip('*') for p in fa.params}

        # ── Walk every AST node ──────────────────────────────────────────
        for node in ast.walk(func_def):

            # Control flow
            if isinstance(node, ast.For):
                fa.has_for_loop = True
            elif isinstance(node, ast.While):
                fa.has_while_loop = True
            elif isinstance(node, ast.If):
                fa.has_conditions = True

            # Recursion detection
            elif isinstance(node, ast.Call):
                if isinstance(node.func, ast.Name):
                    fname = node.func.id
                    if fname == fa.name:
                        fa.is_recursive = True
                    elif fname in PYTHON_BUILTINS:
                        fa.calls_builtins.add(fname)
                    else:
                        fa.calls_external.add(fname)
                elif isinstance(node.func, ast.Attribute):
                    method = node.func.attr
                    # Detect common patterns
                    if method in ('append', 'extend', 'insert', 'remove', 'pop', 'clear', 'sort', 'reverse'):
                        fa.list_mutations = True
                    if method in ('split', 'join', 'strip', 'replace', 'format', 'encode', 'decode',
                                  'upper', 'lower', 'startswith', 'endswith', 'find', 'count'):
                        fa.string_ops = True
                    if method in ('get', 'update', 'keys', 'values', 'items', 'setdefault', 'pop'):
                        fa.dict_access = True
                    if method in ('append', 'extend', 'insert', 'remove', 'pop'):
                        fa.list_mutations = True

            # Raise statements
            elif isinstance(node, ast.Raise):
                if node.exc:
                    try:
                        exc_str = ast.unparse(node.exc)
                        # Extract just the exception class name
                        match = re.match(r'^(\w+)', exc_str)
                        if match:
                            fa.raises_exceptions.append(match.group(1))
                    except Exception:
                        pass

            # Try/except
            elif isinstance(node, ast.Try):
                fa.has_try_except = True
                for handler in node.handlers:
                    if handler.type:
                        try:
                            fa.caught_exceptions.append(ast.unparse(handler.type))
                        except Exception:
                            pass

            # Comprehensions
            elif isinstance(node, ast.ListComp):
                fa.has_list_comp = True
            elif isinstance(node, ast.DictComp):
                fa.has_dict_comp = True
            elif isinstance(node, ast.SetComp):
                fa.has_set_comp = True
            elif isinstance(node, ast.GeneratorExp):
                fa.has_generator = True

            # Yield (generator function)
            elif isinstance(node, (ast.Yield, ast.YieldFrom)):
                fa.has_yield = True

            # Arithmetic / bitwise binary ops
            elif isinstance(node, ast.BinOp):
                op_name = OP_NAMES.get(type(node.op))
                if op_name:
                    if op_name in ('add', 'sub', 'mul', 'div', 'mod', 'pow', 'floor_div', 'matmul'):
                        fa.arithmetic_ops.add(op_name)
                    else:
                        fa.bitwise_ops.add(op_name)

            # Comparisons
            elif isinstance(node, ast.Compare):
                for op in node.ops:
                    op_name = OP_NAMES.get(type(op))
                    if op_name:
                        fa.comparison_ops.add(op_name)

            # Boolean ops
            elif isinstance(node, ast.BoolOp):
                fa.boolean_ops.add('and' if isinstance(node.op, ast.And) else 'or')

            # Return analysis
            elif isinstance(node, ast.Return) and node.value:
                try:
                    expr = ast.unparse(node.value)
                    fa.return_expressions.append(expr)
                    # Detect boolean returns
                    if expr in ('True', 'False') or any(
                        op in fa.comparison_ops for op in ('eq', 'lt', 'gt', 'lte', 'gte', 'neq')
                    ):
                        fa.returns_boolean = True
                    # Detect collection returns
                    if isinstance(node.value, (ast.List, ast.Dict, ast.Set, ast.Tuple)):
                        fa.returns_collection = True
                    # Self return (builder)
                    if expr == 'self':
                        fa.returns_self = True
                except Exception:
                    pass

            # Attribute access — infer input type hints
            elif isinstance(node, ast.Attribute):
                if isinstance(node.value, ast.Name) and node.value.id in param_names:
                    fa.accesses_attributes.append((node.value.id, node.attr))

            # Subscript — dict/list access
            elif isinstance(node, ast.Subscript):
                if isinstance(node.value, ast.Name) and node.value.id in param_names:
                    fa.dict_access = True

        # ── Multiple returns → possible early return pattern ───────────
        returns = [n for n in ast.walk(func_def) if isinstance(n, ast.Return)]
        if len(returns) > 1:
            fa.has_early_return = True

        # ── Infer return type if not annotated ─────────────────────────
        if fa.return_type is None:
            if fa.has_yield:
                fa.return_type = "Generator"
            elif fa.returns_boolean:
                fa.return_type = "bool"
            elif fa.returns_collection:
                fa.return_type = "list | dict | tuple"
            elif fa.return_expressions:
                fa.return_type = "Any"
            else:
                fa.return_type = "None"

        # ── Tag the function's purpose ─────────────────────────────────
        name_lower = fa.name.lower()

        if any(w in name_lower for w in ('validate', 'check', 'verify', 'is_', 'has_', 'can_')):
            fa.purpose_tags.add('validator')
        if any(w in name_lower for w in ('calculate', 'compute', 'calc', 'sum_', 'total', 'average', 'mean')):
            fa.purpose_tags.add('calculator')
        if any(w in name_lower for w in ('parse', 'extract', 'read', 'load', 'fetch', 'get_')):
            fa.purpose_tags.add('extractor')
        if any(w in name_lower for w in ('format', 'convert', 'transform', 'encode', 'decode', 'serialize')):
            fa.purpose_tags.add('transformer')
        if any(w in name_lower for w in ('sort', 'order', 'rank', 'arrange')):
            fa.purpose_tags.add('sorter')
        if any(w in name_lower for w in ('filter', 'exclude', 'include', 'select', 'search', 'find')):
            fa.purpose_tags.add('filter')
        if any(w in name_lower for w in ('save', 'write', 'store', 'persist', 'export', 'dump')):
            fa.purpose_tags.add('writer')
        if any(w in name_lower for w in ('send', 'post', 'notify', 'publish', 'emit', 'dispatch')):
            fa.purpose_tags.add('dispatcher')
        if any(w in name_lower for w in ('delete', 'remove', 'clear', 'clean', 'reset', 'purge')):
            fa.purpose_tags.add('cleaner')
        if any(w in name_lower for w in ('merge', 'combine', 'join', 'concat', 'union', 'aggregate')):
            fa.purpose_tags.add('aggregator')
        if any(w in name_lower for w in ('split', 'chunk', 'partition', 'divide', 'segment')):
            fa.purpose_tags.add('splitter')
        if any(w in name_lower for w in ('generate', 'create', 'build', 'make', 'construct', 'init')):
            fa.purpose_tags.add('generator')
        if fa.is_recursive:
            fa.purpose_tags.add('recursive')
        if fa.has_yield:
            fa.purpose_tags.add('generator_func')
        if fa.has_try_except:
            fa.purpose_tags.add('error_handler')

        # Arithmetic-heavy without name match
        if fa.arithmetic_ops and not fa.purpose_tags:
            fa.purpose_tags.add('calculator')

        # Boolean return without name match
        if fa.returns_boolean and not fa.purpose_tags:
            fa.purpose_tags.add('validator')

        return fa

    # ── Natural-Language Summary ───────────────────────────────────────────

    @staticmethod
    def build_summary(fa: FunctionAnalysis) -> str:
        """Produce a one-sentence description grounded in what the function actually does."""
        name = fa.name
        tags = fa.purpose_tags

        # ── Purpose-driven opening ────────────────────────────────────
        if 'validator' in tags:
            verb = f"Validates whether {_humanise(name)} meets the required criteria"
        elif 'calculator' in tags:
            verb = f"Computes {_humanise(name)}"
        elif 'extractor' in tags:
            verb = f"Extracts and returns {_humanise(name)}"
        elif 'transformer' in tags:
            verb = f"Converts or transforms {_humanise(name)}"
        elif 'sorter' in tags:
            verb = f"Sorts and returns {_humanise(name)}"
        elif 'filter' in tags:
            verb = f"Filters {_humanise(name)} based on given criteria"
        elif 'writer' in tags:
            verb = f"Persists {_humanise(name)} to the target destination"
        elif 'dispatcher' in tags:
            verb = f"Dispatches {_humanise(name)} to the appropriate handler"
        elif 'cleaner' in tags:
            verb = f"Removes or resets {_humanise(name)}"
        elif 'aggregator' in tags:
            verb = f"Combines multiple inputs into {_humanise(name)}"
        elif 'splitter' in tags:
            verb = f"Splits {_humanise(name)} into smaller parts"
        elif 'generator_func' in tags:
            verb = f"Lazily yields values from {_humanise(name)}"
        elif 'generator' in tags:
            verb = f"Constructs and returns {_humanise(name)}"
        elif 'recursive' in tags:
            verb = f"Recursively processes {_humanise(name)}"
        elif 'error_handler' in tags:
            verb = f"Safely executes {_humanise(name)} with error handling"
        else:
            verb = f"Performs {_humanise(name)}"

        # ── Structural qualifiers ─────────────────────────────────────
        qualifiers = []

        if fa.has_conditions and fa.has_for_loop:
            qualifiers.append("iterating over elements with conditional branching")
        elif fa.has_for_loop:
            qualifiers.append("iterating over the provided input")
        elif fa.has_while_loop:
            qualifiers.append("using an iterative loop until a condition is met")
        elif fa.has_conditions:
            qualifiers.append("applying conditional logic")

        if fa.has_list_comp:
            qualifiers.append("via a list comprehension")
        elif fa.has_dict_comp:
            qualifiers.append("via a dict comprehension")
        elif fa.has_generator:
            qualifiers.append("using a generator expression")

        if fa.is_recursive:
            qualifiers.append("through recursive calls")

        if fa.arithmetic_ops:
            ops = sorted(fa.arithmetic_ops)
            readable = {
                'add': 'addition', 'sub': 'subtraction', 'mul': 'multiplication',
                'div': 'division', 'mod': 'modulo', 'pow': 'exponentiation',
                'floor_div': 'floor division', 'matmul': 'matrix multiplication'
            }
            op_list = [readable.get(o, o) for o in ops[:2]]
            qualifiers.append(f"using {' and '.join(op_list)}")

        if fa.string_ops:
            qualifiers.append("with string manipulation")

        # ── Build sentence ────────────────────────────────────────────
        if qualifiers:
            sentence = f"{verb}, {', '.join(qualifiers[:2])}."
        else:
            sentence = f"{verb}."

        return sentence.capitalize()

    # ── Parameter Description ──────────────────────────────────────────────

    @staticmethod
    def describe_param(param: dict, fa: FunctionAnalysis) -> str:
        """Generate a meaningful, specific description for each parameter."""
        name = param['name'].lstrip('*')
        ptype = param['type']
        tags = fa.purpose_tags

        # Attributes accessed on this param give strong type hints
        attrs = [attr for pname, attr in fa.accesses_attributes if pname == name]

        # Common semantic name patterns
        name_lower = name.lower()

        if name == 'self':
            return "The instance of the class."
        if name == 'cls':
            return "The class itself (used in class methods)."

        # Type-guided
        if ptype in ('str', 'String'):
            if any(w in name_lower for w in ('path', 'file', 'dir', 'url', 'uri')):
                return f"File system path or URL string to be processed."
            if any(w in name_lower for w in ('query', 'search', 'keyword', 'pattern')):
                return f"Search query or pattern string."
            if any(w in name_lower for w in ('name', 'title', 'label')):
                return f"Human-readable name or label."
            if any(w in name_lower for w in ('msg', 'message', 'text', 'content', 'body')):
                return f"Text content or message body."
            return f"String input value."

        if ptype in ('int', 'Integer'):
            if any(w in name_lower for w in ('count', 'num', 'n', 'size', 'length', 'limit')):
                return f"Integer count or size constraint."
            if any(w in name_lower for w in ('index', 'idx', 'pos', 'position', 'offset')):
                return f"Zero-based index or positional offset."
            if any(w in name_lower for w in ('max', 'min', 'cap', 'threshold', 'bound')):
                return f"Numeric bound or threshold."
            if any(w in name_lower for w in ('id', '_id', 'pk')):
                return f"Unique integer identifier."
            return f"Integer numeric value."

        if ptype in ('float', 'Float'):
            if any(w in name_lower for w in ('rate', 'ratio', 'factor', 'weight', 'prob', 'probability')):
                return f"Decimal rate or ratio, typically in the range [0.0, 1.0]."
            if any(w in name_lower for w in ('price', 'cost', 'amount', 'total', 'fee', 'tax')):
                return f"Monetary amount or financial value."
            if any(w in name_lower for w in ('threshold', 'tolerance', 'epsilon', 'delta')):
                return f"Floating-point tolerance or precision threshold."
            return f"Floating-point numeric value."

        if ptype in ('bool', 'Boolean'):
            if any(w in name_lower for w in ('verbose', 'debug', 'log')):
                return f"If True, enables verbose logging or debug output."
            if any(w in name_lower for w in ('strict', 'enforce', 'required')):
                return f"If True, enforces strict validation rules."
            if any(w in name_lower for w in ('reverse', 'desc', 'descending')):
                return f"If True, reverses the output order."
            if any(w in name_lower for w in ('inplace', 'in_place', 'copy')):
                return f"If True, modifies the object in place instead of returning a copy."
            return f"Boolean flag controlling behavior."

        if ptype.startswith('list') or ptype.startswith('List') or ptype == 'Sequence':
            if any(w in name_lower for w in ('items', 'elements', 'entries', 'records', 'rows')):
                return f"Collection of items to be processed."
            if any(w in name_lower for w in ('keys', 'fields', 'columns', 'attrs')):
                return f"List of field names or keys to consider."
            return f"List of input elements."

        if ptype.startswith('dict') or ptype.startswith('Dict') or ptype == 'Mapping':
            if any(w in name_lower for w in ('config', 'cfg', 'settings', 'options', 'params')):
                return f"Configuration dictionary mapping option names to their values."
            if any(w in name_lower for w in ('kwargs', 'kwds')):
                return f"Additional keyword arguments passed through."
            return f"Dictionary of key-value pairs."

        if ptype.startswith('Optional'):
            inner = ptype[9:-1] if ptype.startswith('Optional[') else 'Any'
            return f"Optional {inner} value; if None, a default behaviour is applied."

        if ptype.startswith('Callable') or ptype == 'callable':
            if any(w in name_lower for w in ('key', 'keyfunc', 'sort_key')):
                return f"Callable used as a sort key; receives one element and returns a comparable value."
            if any(w in name_lower for w in ('callback', 'handler', 'hook', 'func', 'fn')):
                return f"Callable invoked on each element or event; receives the element as its argument."
            if any(w in name_lower for w in ('predicate', 'condition', 'cond')):
                return f"Predicate function returning True for elements that should be selected."
            return f"Callable (function or lambda) applied during processing."

        if ptype in ('Any', None, 'any'):
            # Fallback to name-based inference
            if any(w in name_lower for w in ('data', 'payload', 'body', 'content')):
                return f"Input data to be processed."
            if any(w in name_lower for w in ('obj', 'object', 'instance', 'item', 'element')):
                return f"Target object or element."
            if any(w in name_lower for w in ('value', 'val', 'v')):
                return f"Input value to be evaluated or transformed."
            if any(w in name_lower for w in ('source', 'src', 'origin', 'input')):
                return f"Source data or input to the operation."
            if any(w in name_lower for w in ('target', 'dest', 'destination', 'output')):
                return f"Destination or target for the result."
            if attrs:
                return f"Object with attributes: {', '.join(f'.{a}' for a in attrs[:3])}."

        # Attribute-based fallback
        if attrs:
            return f"{ptype} instance; uses .{attrs[0]} during processing."

        return f"{ptype} value used as input."

    # ── Return Description ─────────────────────────────────────────────────

    @staticmethod
    def describe_return(fa: FunctionAnalysis) -> str:
        """Build a meaningful return value description."""
        rtype = fa.return_type or 'Any'
        tags = fa.purpose_tags
        name_lower = fa.name.lower()

        if rtype == 'None' and not fa.has_yield:
            return "This function performs side effects and does not return a value."

        if fa.has_yield:
            return f"Yields individual elements lazily one at a time."

        if 'validator' in tags or fa.returns_boolean:
            return f"True if the input satisfies the condition, False otherwise."

        if 'calculator' in tags:
            if 'mul' in fa.arithmetic_ops or 'pow' in fa.arithmetic_ops:
                return f"The computed {_humanise(name_lower)} as a numeric value."
            return f"The numeric result of the computation."

        if 'extractor' in tags:
            return f"The extracted value, or None if not found."

        if 'transformer' in tags:
            return f"The transformed result as {rtype}."

        if 'sorter' in tags:
            return f"A new sorted sequence of the same element type."

        if 'filter' in tags:
            return f"A filtered collection containing only elements that match the criteria."

        if 'aggregator' in tags:
            return f"The merged or combined result."

        if 'splitter' in tags:
            return f"A list of sub-parts produced by the split."

        if 'generator_func' in tags:
            return f"A generator that yields elements one by one."

        if 'generator' in tags:
            return f"A new {rtype} instance constructed from the provided inputs."

        if 'error_handler' in tags:
            return f"The result of the operation if successful, or a handled default on failure."

        if rtype.startswith('list') or rtype.startswith('List'):
            return f"A list containing the processed results."

        if rtype.startswith('dict') or rtype.startswith('Dict'):
            return f"A dictionary mapping keys to the computed values."

        if rtype in ('bool',):
            return f"True if the condition holds, False otherwise."

        if rtype in ('str', 'String'):
            return f"The resulting string after processing."

        if rtype in ('int', 'float', 'Integer', 'Float'):
            return f"The numeric result of the operation."

        return f"The result of the operation as {rtype}."

    # ── Raises Section ─────────────────────────────────────────────────────

    @staticmethod
    def describe_raises(fa: FunctionAnalysis) -> list:
        """Return a list of (ExceptionType, description) tuples."""
        results = []
        seen = set()
        descriptions = {
            'ValueError': "If the input value is out of the expected range or has an invalid format.",
            'TypeError': "If an argument has an incompatible type.",
            'KeyError': "If a required key is not present in the mapping.",
            'IndexError': "If an index is out of bounds.",
            'AttributeError': "If the object does not have the required attribute.",
            'RuntimeError': "If an unexpected error occurs during execution.",
            'NotImplementedError': "If this method must be overridden by a subclass.",
            'PermissionError': "If the operation lacks the necessary file system permissions.",
            'FileNotFoundError': "If the specified file or path does not exist.",
            'OSError': "If an OS-level I/O error occurs.",
            'StopIteration': "When the iterator is exhausted.",
            'OverflowError': "If the computed value exceeds numeric limits.",
            'ZeroDivisionError': "If a divisor of zero is encountered.",
        }
        for exc in fa.raises_exceptions:
            if exc not in seen:
                seen.add(exc)
                desc = descriptions.get(exc, "If an error condition specific to this function is met.")
                results.append((exc, desc))
        return results

    # ── Docstring Builders ─────────────────────────────────────────────────

    def generate_google_docstring(self, fa: FunctionAnalysis) -> str:
        summary = self.build_summary(fa)
        doc = f'    """{summary}\n'

        visible_params = [p for p in fa.params if p['name'] != 'self' and p['name'] != 'cls']
        if visible_params:
            doc += '\n    Args:\n'
            for p in visible_params:
                desc = self.describe_param(p, fa)
                default = f" Defaults to {p['default']}." if p.get('default') else ""
                doc += f"        {p['name']} ({p['type']}): {desc}{default}\n"

        doc += f"\n    Returns:\n        {fa.return_type}: {self.describe_return(fa)}\n"

        raises = self.describe_raises(fa)
        if raises:
            doc += '\n    Raises:\n'
            for exc, desc in raises:
                doc += f"        {exc}: {desc}\n"

        if fa.is_recursive:
            doc += '\n    Note:\n        This function uses recursion. Ensure the base case is always reachable\n        to avoid maximum recursion depth errors.\n'
        if fa.has_yield:
            doc += '\n    Note:\n        This is a generator function. Use next() or iterate with a for-loop.\n'

        doc += '    """\n'
        return doc

    def generate_numpy_docstring(self, fa: FunctionAnalysis) -> str:
        summary = self.build_summary(fa)
        doc = f'    """{summary}\n'

        visible_params = [p for p in fa.params if p['name'] != 'self' and p['name'] != 'cls']
        if visible_params:
            doc += '\n    Parameters\n    ----------\n'
            for p in visible_params:
                desc = self.describe_param(p, fa)
                default = f", default {p['default']}" if p.get('default') else ""
                doc += f"    {p['name']} : {p['type']}{default}\n"
                doc += f"        {desc}\n"

        doc += f"\n    Returns\n    -------\n    {fa.return_type}\n"
        doc += f"        {self.describe_return(fa)}\n"

        raises = self.describe_raises(fa)
        if raises:
            doc += '\n    Raises\n    ------\n'
            for exc, desc in raises:
                doc += f"    {exc}\n        {desc}\n"

        notes = []
        if fa.is_recursive:
            notes.append("Uses recursion. Ensure the base case terminates correctly.")
        if fa.has_yield:
            notes.append("Generator function — iterate with a for-loop or call next().")
        if notes:
            doc += '\n    Notes\n    -----\n'
            for n in notes:
                doc += f"    {n}\n"

        doc += '    """\n'
        return doc


# ── Utility ────────────────────────────────────────────────────────────────

def _humanise(name: str) -> str:
    """Convert snake_case function name to readable English phrase."""
    # Strip common prefixes
    for prefix in ('get_', 'set_', 'is_', 'has_', 'can_', 'do_', 'run_', 'calc_', 'compute_'):
        if name.startswith(prefix):
            name = name[len(prefix):]
            break
    return name.replace('_', ' ').strip()


# ── Top-level orchestration ────────────────────────────────────────────────

analyzer = DocGenieAnalyzer()
generation_history = []


def generate_docstring(code: str, style: str = "google"):
    if not code or not code.strip():
        return None, "⚠  Paste a Python function above to get started."

    try:
        signature, func_def = analyzer.extract_signature(code)
        if not signature or func_def is None:
            return None, "⚠  Could not parse the function — check for syntax errors."

        fa = analyzer.analyze(func_def, signature)

        docstring = (
            analyzer.generate_google_docstring(fa)
            if style.lower() == "google"
            else analyzer.generate_numpy_docstring(fa)
        )

        lines = code.split('\n')
        # Insert docstring right after the def line
        final_code = lines[0] + '\n' + docstring + '\n'.join(lines[1:]) + '\n'

        generation_history.append({
            'code': code, 'docstring': docstring,
            'style': style, 'timestamp': datetime.now().isoformat(),
            'name': fa.name, 'tags': list(fa.purpose_tags),
        })

        tag_str = ', '.join(sorted(fa.purpose_tags)) if fa.purpose_tags else 'general'
        return final_code, f"✓  {style.capitalize()} docstring generated [{tag_str}] — {len(generation_history)} in session."

    except Exception as e:
        return None, f"✗  Internal error: {str(e)}"


def process_upload(file):
    if not file:
        return None, "No file selected."
    try:
        with open(file.name, 'r', encoding='utf-8') as f:
            content = f.read()
        return content, f"✓  Loaded: {Path(file.name).name}"
    except Exception as e:
        return None, f"✗  Could not read file: {str(e)}"


def clear_history():
    global generation_history
    generation_history = []
    return "✓  Session history cleared."


def download_pdf():
    if not generation_history:
        return "⚠  Generate some docstrings first.", None

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"doc_genie_{timestamp}.pdf"

    try:
        doc = SimpleDocTemplate(filename, pagesize=A4, topMargin=0.6*inch, bottomMargin=0.6*inch)
        styles = getSampleStyleSheet()
        story = []

        header_style = ParagraphStyle('H', parent=styles['Heading1'], fontSize=20, spaceAfter=6)
        code_style = ParagraphStyle('C', parent=styles['Normal'], fontSize=8, fontName='Courier',
                                    backColor='#F5F5F5', borderColor='#DDDDDD', borderWidth=1, borderPadding=8)

        story.append(Paragraph("Doc-Genie — Documentation Report", header_style))
        story.append(Paragraph(f"Generated {datetime.now().strftime('%B %d, %Y at %H:%M')}", styles['Normal']))
        story.append(Spacer(1, 0.3*inch))

        for i, gen in enumerate(generation_history, 1):
            name = gen.get('name', f'Function #{i}')
            tags = ', '.join(gen.get('tags', [])) or '—'
            story.append(Paragraph(f"#{i}  {name}  ({gen['style'].upper()})", styles['Heading2']))
            story.append(Paragraph(f"Detected purpose: {tags}", styles['Italic']))
            story.append(Spacer(1, 0.1*inch))
            story.append(Paragraph("Original Code:", styles['Normal']))
            story.append(Preformatted(gen['code'], code_style))
            story.append(Spacer(1, 0.1*inch))
            story.append(Paragraph("Generated Docstring:", styles['Normal']))
            story.append(Preformatted(gen['docstring'], code_style))
            story.append(Spacer(1, 0.25*inch))
            if i % 2 == 0 and i != len(generation_history):
                story.append(PageBreak())

        story.append(Paragraph(f"Total: {len(generation_history)} function(s) documented.", styles['Normal']))
        doc.build(story)
        return f"✓  PDF saved: {filename}", filename
    except Exception as e:
        return f"✗  Export failed: {str(e)}", None


def download_txt():
    if not generation_history:
        return "⚠  Generate some docstrings first.", None

    content = f"Doc-Genie Documentation Report\nGenerated: {datetime.now()}\n{'─'*60}\n\n"
    for i, gen in enumerate(generation_history, 1):
        tags = ', '.join(gen.get('tags', [])) or '—'
        content += f"[ #{i} | {gen.get('name','?')} | {gen['style'].upper()} | purpose: {tags} ]\n\n"
        content += f"ORIGINAL CODE:\n{gen['code']}\n\n"
        content += f"GENERATED DOCSTRING:\n{gen['docstring']}\n"
        content += f"{'─'*60}\n\n"

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"doc_genie_{timestamp}.txt"
    try:
        Path(filename).write_text(content)
        return f"✓  TXT saved: {filename}", filename
    except Exception as e:
        return f"✗  Export failed: {str(e)}", None


def generate_whatsapp_link():
    if not generation_history:
        return "⚠  Nothing to share yet."
    message = "📋 *Doc-Genie Report*\n\n"
    for i, gen in enumerate(generation_history[-3:], 1):
        tags = ', '.join(gen.get('tags', []))
        message += f"*#{i} {gen.get('name','?')}* ({tags})\n`{gen['code'][:50]}...`\n\n"
    link = f"https://wa.me/?text={urllib.parse.quote(message)}"
    return f"WhatsApp link ready:\n\n{link}"


def generate_facebook_link():
    if not generation_history:
        return "⚠  Nothing to share yet."
    msg = f"Just documented {len(generation_history)} Python function(s) using Doc-Genie!"
    link = f"https://www.facebook.com/sharer/sharer.php?quote={urllib.parse.quote(msg)}"
    return f"Facebook link ready:\n\n{link}"


# ============================================================================
# UI — Editorial Theme
# ============================================================================

CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=DM+Mono:wght@300;400;500&family=Syne:wght@400;600;700;800&family=DM+Sans:wght@300;400;500&display=swap');

:root {
    --ink: #0D0D0D;
    --paper: #F7F4EF;
    --cream: #EDE8E0;
    --sand: #D4CDBF;
    --accent: #C84B31;
    --accent2: #3B5BA5;
    --muted: #7A7167;
    --code-bg: #1A1A18;
    --code-fg: #E8E0D0;
    --border: #C8C0B0;
}

* { box-sizing: border-box; }

body, .gradio-container {
    background: var(--paper) !important;
    font-family: 'DM Sans', sans-serif !important;
    color: var(--ink) !important;
}

.app-header {
    background: var(--ink);
    padding: 28px 40px 22px;
    margin: -12px -12px 32px -12px;
    display: flex;
    align-items: baseline;
    gap: 16px;
    border-bottom: 3px solid var(--accent);
}
.app-header h1 {
    font-family: 'Syne', sans-serif !important;
    font-weight: 800 !important;
    font-size: 2rem !important;
    color: #F7F4EF !important;
    letter-spacing: -0.02em !important;
    margin: 0 !important;
}
.app-header .tagline {
    font-family: 'DM Mono', monospace !important;
    font-size: 0.72rem !important;
    color: var(--sand) !important;
    letter-spacing: 0.08em !important;
    text-transform: uppercase !important;
}

.section-label {
    font-family: 'DM Mono', monospace !important;
    font-size: 0.65rem !important;
    font-weight: 500 !important;
    letter-spacing: 0.12em !important;
    text-transform: uppercase !important;
    color: var(--muted) !important;
    border-bottom: 1px solid var(--border) !important;
    padding-bottom: 6px !important;
    margin-bottom: 12px !important;
}

.gradio-code textarea, .gradio-code .cm-editor, code, pre {
    font-family: 'DM Mono', monospace !important;
    font-size: 0.82rem !important;
    background: var(--code-bg) !important;
    color: var(--code-fg) !important;
    border-radius: 2px !important;
    border: none !important;
}
.input-code .cm-editor {
    background: #FAFAF8 !important;
    color: var(--ink) !important;
    border: 1px solid var(--border) !important;
}

button, .gr-button {
    font-family: 'DM Mono', monospace !important;
    font-size: 0.75rem !important;
    letter-spacing: 0.06em !important;
    text-transform: uppercase !important;
    border-radius: 2px !important;
    transition: all 0.15s ease !important;
    cursor: pointer !important;
}
.btn-generate {
    background: var(--accent) !important;
    color: white !important;
    border: none !important;
    padding: 12px 24px !important;
    font-weight: 500 !important;
}
.btn-generate:hover { background: #A63A24 !important; transform: translateY(-1px) !important; }

.btn-secondary {
    background: transparent !important;
    color: var(--ink) !important;
    border: 1px solid var(--border) !important;
    padding: 9px 16px !important;
}
.btn-secondary:hover { border-color: var(--ink) !important; background: var(--cream) !important; }

.btn-accent2 {
    background: var(--accent2) !important;
    color: white !important;
    border: none !important;
    padding: 9px 16px !important;
}
.btn-accent2:hover { background: #2E4A8A !important; }

.btn-danger {
    background: transparent !important;
    color: var(--accent) !important;
    border: 1px solid var(--accent) !important;
    padding: 9px 16px !important;
}
.btn-danger:hover { background: var(--accent) !important; color: white !important; }

.status-bar textarea {
    font-family: 'DM Mono', monospace !important;
    font-size: 0.74rem !important;
    background: var(--cream) !important;
    border: none !important;
    border-left: 3px solid var(--accent) !important;
    border-radius: 0 2px 2px 0 !important;
    color: var(--muted) !important;
    padding: 8px 12px !important;
}

.sidebar-rule { border: none !important; border-top: 1px solid var(--border) !important; margin: 18px 0 !important; }
.main-col { padding-right: 20px !important; }

.gr-file {
    border: 1.5px dashed var(--border) !important;
    border-radius: 2px !important;
    background: var(--cream) !important;
}
.gr-file:hover { border-color: var(--accent) !important; background: white !important; }

::-webkit-scrollbar { width: 4px; height: 4px; }
::-webkit-scrollbar-track { background: var(--cream); }
::-webkit-scrollbar-thumb { background: var(--sand); border-radius: 2px; }

.gradio-label, label {
    font-family: 'DM Sans', sans-serif !important;
    font-size: 0.8rem !important;
    font-weight: 500 !important;
    color: var(--muted) !important;
    text-transform: uppercase !important;
    letter-spacing: 0.06em !important;
}
"""

print("⟳  Initializing Doc-Genie v3...")

with gr.Blocks(title="Doc-Genie v3 — Python Docstring Generator", css=CUSTOM_CSS) as demo:

    gr.HTML("""
    <div class="app-header">
        <h1>Doc-Genie</h1>
        <span class="tagline">Python Docstring Generator · v3.0 — Deep Contextual Analysis</span>
    </div>
    """)

    with gr.Row(equal_height=False):

        with gr.Column(scale=5, elem_classes="main-col"):

            gr.HTML('<p class="section-label">01 — Input</p>')

            code_input = gr.Code(
                language="python",
                label="Paste your Python function here",
                lines=12,
                elem_classes="input-code",
                value=(
                    "def calculate_discount(price: float, rate: float = 0.1) -> float:\n"
                    "    if rate < 0 or rate > 1:\n"
                    "        raise ValueError('Rate must be between 0 and 1')\n"
                    "    discount = price * rate\n"
                    "    return price - discount"
                )
            )

            with gr.Row():
                file_upload = gr.File(label="Or upload a .py file", file_types=['.py'], scale=2)
                upload_status = gr.Textbox(
                    label="File status", interactive=False, lines=2,
                    scale=1, elem_classes="status-bar"
                )

            def process_and_display(file):
                if file:
                    content, status = process_upload(file)
                    return content, status
                return "", "—"
            file_upload.change(process_and_display, inputs=[file_upload], outputs=[code_input, upload_status])

            gr.HTML('<div class="sidebar-rule"></div>')
            gr.HTML('<p class="section-label">02 — Style & Generate</p>')

            with gr.Row():
                docstring_style = gr.Radio(["google", "numpy"], value="google", label="Docstring format", scale=1)
                generate_btn = gr.Button("→  Generate Docstring", scale=2, elem_classes="btn-generate")

            gr.HTML('<div class="sidebar-rule"></div>')
            gr.HTML('<p class="section-label">03 — Output</p>')

            code_output = gr.Code(language="python", label="Function with docstring", lines=18, interactive=False)
            gen_status = gr.Textbox(label="Status", interactive=False, lines=1, elem_classes="status-bar")

            generate_btn.click(
                generate_docstring,
                inputs=[code_input, docstring_style],
                outputs=[code_output, gen_status]
            )

        with gr.Column(scale=2):

            gr.HTML('<p class="section-label">Session</p>')
            clear_btn = gr.Button("✕  Clear History", elem_classes="btn-danger")
            session_status = gr.Textbox(label="", interactive=False, lines=1, elem_classes="status-bar")
            clear_btn.click(clear_history, outputs=[session_status])

            gr.HTML('<div class="sidebar-rule"></div>')
            gr.HTML('<p class="section-label">Export</p>')

            pdf_btn = gr.Button("↓  Download PDF", elem_classes="btn-accent2")
            txt_btn = gr.Button("↓  Download TXT", elem_classes="btn-secondary")
            pdf_file = gr.File(label="PDF", visible=True)
            txt_file = gr.File(label="TXT", visible=True)
            export_status = gr.Textbox(label="Export status", interactive=False, lines=2, elem_classes="status-bar")
            pdf_btn.click(download_pdf, outputs=[export_status, pdf_file])
            txt_btn.click(download_txt, outputs=[export_status, txt_file])

            gr.HTML('<div class="sidebar-rule"></div>')
            gr.HTML('<p class="section-label">Share</p>')

            whatsapp_btn = gr.Button("↗  WhatsApp", elem_classes="btn-secondary")
            facebook_btn = gr.Button("↗  Facebook", elem_classes="btn-secondary")
            share_output = gr.Textbox(label="Share link", interactive=False, lines=3, elem_classes="status-bar")
            whatsapp_btn.click(generate_whatsapp_link, outputs=[share_output])
            facebook_btn.click(generate_facebook_link, outputs=[share_output])

            gr.HTML('<div class="sidebar-rule"></div>')
            gr.HTML("""
            <div style="font-family:'DM Mono',monospace; font-size:0.68rem; color:#7A7167; line-height:2;">
                <span style="color:#0D0D0D;font-weight:600;">v3 Detects</span><br>
                Purpose tags · Recursion<br>
                Generators · Try/except<br>
                Comprehensions · Raises<br>
                Arithmetic ops · String ops<br>
                Type-aware param docs<br>
                Inferred return type<br><br>
                <span style="color:#0D0D0D;font-weight:600;">Formats</span><br>
                Google · NumPy
            </div>
            """)

print("✓  Doc-Genie v3 ready — http://localhost:7860\n")
demo.launch(share=True, show_error=True)

⟳  Initializing Doc-Genie v3...


/tmp/ipykernel_477/3445889477.py:1031: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="Doc-Genie v3 — Python Docstring Generator", css=CUSTOM_CSS) as demo:


✓  Doc-Genie v3 ready — http://localhost:7860

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9296c4dc9dc9c97402.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
